# 🤖 Trenowanie Modeli - Moodify

## 🎯 Cel
Wytrenowanie dwóch prostych modeli do rozpoznawania emocji:
- 📝 **Model Tekstowy** - scikit-learn (TF-IDF + klasyfikator)
- 📸 **Model Obrazowy** - TensorFlow/Keras (Transfer Learning)

---
# CZĘŚĆ 1: MODEL TEKSTOWY (scikit-learn)
---

## 1. Instalacja bibliotek

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn

## 2. Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

## 3. Wczytanie danych tekstowych

In [ ]:
train_text = pd.read_csv('processed_data/train_text.csv')
val_text = pd.read_csv('processed_data/val_text.csv')
test_text = pd.read_csv('processed_data/test_text.csv')

print(f"Train: {len(train_text)}, Val: {len(val_text)}, Test: {len(test_text)}")
train_text.head()

## 4. TF-IDF Vectorization

**Parametry:**
- max_features=5000: Top 5000 słów
- min_df=2: Słowo w min 2 dokumentach
- max_df=0.8: Ignoruj słowa w >80% dokumentów
- ngram_range=(1,2): Pojedyncze słowa i pary

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.8, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_text['text'])
y_train = train_text['emotion']

X_val = vectorizer.transform(val_text['text'])
y_val = val_text['emotion']

X_test = vectorizer.transform(test_text['text'])
y_test = test_text['emotion']

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Słownik: {len(vectorizer.vocabulary_)} słów")

## 5. Trening modelu SVM

In [ ]:
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)

val_accuracy = svm.score(X_val, y_val)
print(f"Validation Accuracy: {val_accuracy:.2%}")

## 6. Ewaluacja na zbiorze testowym

In [ ]:
y_pred = svm.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)

print(f"Test Accuracy: {report['accuracy']:.2%}\n")
for emotion, metrics in report.items():
    if emotion not in ['accuracy', 'macro avg', 'weighted avg']:
        print(f"{emotion}: Precision={metrics['precision']:.2%}, Recall={metrics['recall']:.2%}, F1={metrics['f1-score']:.2%}")

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
emotions = sorted(train_text['emotion'].unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=emotions,
            yticklabels=emotions)
plt.title('Confusion Matrix - SVM (znormalizowana)', fontsize=16, fontweight='bold')
plt.ylabel('Prawdziwa Emocja', fontsize=12)
plt.xlabel('Przewidziana Emocja', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Zapisanie modelu tekstowego

In [ ]:
import os

os.makedirs('saved_models', exist_ok=True)
joblib.dump(svm, 'saved_models/svm_text_model.pkl')
joblib.dump(vectorizer, 'saved_models/tfidf_vectorizer.pkl')

print("✅ Model zapisany")

## 9. Test predykcji na nowych tekstach

In [ ]:
def predict_emotion_text(text):
    return svm.predict(vectorizer.transform([text]))[0]

test_texts = [
    "I am so happy today!",
    "This is terrible, I hate it",
    "I'm feeling really sad and lonely",
    "Wow, I didn't expect that!",
    "I'm scared of what might happen"
]

for text in test_texts:
    print(f'"{text}" → {predict_emotion_text(text)}')

---
# CZĘŚĆ 2: MODEL OBRAZOWY (TensorFlow/Keras)
---

## 10. Instalacja bibliotek

## 11. Import bibliotek TensorFlow

In [ ]:
!pip install tensorflow[and-cuda] pillow keras-tuner

## 12. Konfiguracja CUDA

⚠️ Wykonać PRZED importem TensorFlow!

In [ ]:
import os
import sys
from pathlib import Path
import ctypes

site_packages = None
for path in sys.path:
    if 'site-packages' in path:
        site_packages = Path(path)
        break

if site_packages:
    nvidia_libs_dirs = [
        site_packages / 'nvidia' / 'cuda_runtime' / 'lib',
        site_packages / 'nvidia' / 'cudnn' / 'lib', 
        site_packages / 'nvidia' / 'cublas' / 'lib',
        site_packages / 'nvidia' / 'cufft' / 'lib',
        site_packages / 'nvidia' / 'curand' / 'lib',
        site_packages / 'nvidia' / 'cusolver' / 'lib',
        site_packages / 'nvidia' / 'cusparse' / 'lib',
        site_packages / 'nvidia' / 'nvjitlink' / 'lib',
    ]
    
    new_paths = [str(p) for p in nvidia_libs_dirs if p.exists()]
    if new_paths:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(new_paths) + (':' + os.environ.get('LD_LIBRARY_PATH', ''))
        for lib_dir in nvidia_libs_dirs:
            if lib_dir.exists():
                for so_file in lib_dir.glob('*.so*'):
                    if so_file.is_file() and not so_file.is_symlink():
                        try:
                            ctypes.CDLL(str(so_file), mode=ctypes.RTLD_GLOBAL)
                            break
                        except:
                            pass

print("✅ CUDA skonfigurowane")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from PIL import Image

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 13. Konfiguracja GPU

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU: {len(gpus)} urządzeń")
else:
    print("⚠️ Brak GPU - trening na CPU")

## 14. Wczytanie danych obrazowych

In [ ]:
train_img = pd.read_csv('processed_data/train_images.csv')
val_img = pd.read_csv('processed_data/val_images.csv')
test_img = pd.read_csv('processed_data/test_images.csv')

train_img = train_img[train_img['emotion'] != 'neutral'].reset_index(drop=True)
val_img = val_img[val_img['emotion'] != 'neutral'].reset_index(drop=True)
test_img = test_img[test_img['emotion'] != 'neutral'].reset_index(drop=True)

emotions_list = sorted(train_img['emotion'].unique())
emotion_to_id = {emotion: idx for idx, emotion in enumerate(emotions_list)}
id_to_emotion = {idx: emotion for emotion, idx in emotion_to_id.items()}

print(f"Train: {len(train_img)}, Val: {len(val_img)}, Test: {len(test_img)}")
print(f"Emocje: {emotions_list}")

## 15. Wczytanie ścieżki bazowej

In [ ]:
import json

with open('processed_data/data_paths.json', 'r') as f:
    data_paths = json.load(f)

BASE_PATH = data_paths['affectnet_base_path']
print(f"Ścieżka bazowa: {BASE_PATH}")

## 16. Generator danych

**Data augmentation:** flip, brightness, contrast, saturation, rotation, zoom, crop  
**Normalizacja:** [0, 1]  
**Labels:** one-hot encoding

In [ ]:
from tensorflow.keras.utils import to_categorical

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5

def create_dataset(df, emotion_to_id, base_path, augment=False):
    def load_image(path, label):
        full_path = tf.strings.join([base_path, os.sep, path])
        img = tf.io.read_file(full_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img / 255.0, label
    
    def augment_image(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.2)
        img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
        img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
        
        if tf.random.uniform([]) > 0.5:
            k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
            img = tf.image.rot90(img, k=k % 4)
        
        zoom_factor = tf.random.uniform([], minval=0.9, maxval=1.1)
        new_size = tf.cast(IMG_SIZE * zoom_factor, tf.int32)
        img = tf.image.resize(img, [new_size, new_size])
        img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE, IMG_SIZE)
        img = tf.image.random_crop(img, size=[IMG_SIZE, IMG_SIZE, 3])
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return tf.clip_by_value(img, 0.0, 1.0), label
    
    labels = to_categorical(df['emotion'].map(emotion_to_id).values, num_classes=NUM_CLASSES)
    dataset = tf.data.Dataset.from_tensor_slices((df['path'].values, labels))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.shuffle(1000)
    
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_img, emotion_to_id, BASE_PATH, augment=True)
val_ds = create_dataset(val_img, emotion_to_id, BASE_PATH, augment=False)
test_ds = create_dataset(test_img, emotion_to_id, BASE_PATH, augment=False)

print("✅ Datasety gotowe")

## 17. Hyperparameter Tuning - Hyperband

**Parametry:** units, dropout, learning rate, L2 reg, fine-tuning CNN  
**Czas:** ~4h, ~60-80 modeli

In [ ]:
import keras_tuner as kt
from tensorflow.keras import regularizers
import datetime

def build_tuned_model(hp):
    base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    
    unfreeze_layers = hp.Int('unfreeze_layers', min_value=0, max_value=50, step=10)
    base_model.trainable = False
    if unfreeze_layers > 0:
        base_model.trainable = True
        for layer in base_model.layers[:-unfreeze_layers]:
            layer.trainable = False
    
    units_1 = hp.Int('units_1', min_value=128, max_value=512, step=64)
    units_2 = hp.Int('units_2', min_value=64, max_value=256, step=32)
    dropout_1 = hp.Float('dropout_1', min_value=0.3, max_value=0.6, step=0.1)
    dropout_2 = hp.Float('dropout_2', min_value=0.2, max_value=0.5, step=0.1)
    learning_rate = hp.Float('learning_rate', min_value=1e-5, max_value=1e-3, sampling='log')
    l2_reg = hp.Float('l2_reg', min_value=0.0001, max_value=0.01, sampling='log')
    
    if unfreeze_layers > 0:
        base_lr_multiplier = hp.Float('base_lr_multiplier', min_value=0.01, max_value=0.1, step=0.01)
    
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(units_1, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_1),
        Dense(units_2, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_2),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

log_dir = "keras_tuner/tensorboard_logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq='epoch', profile_batch=0)

print(f"TensorBoard: tensorboard --logdir keras_tuner/tensorboard_logs")

tuner = kt.Hyperband(
    build_tuned_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    hyperband_iterations=2,
    directory='keras_tuner',
    project_name='moodify_hyperband_4hours',
    overwrite=True,
    seed=42
)

early_stop_tuning = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1)
reduce_lr_tuning = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

print("🚀 Rozpoczynam tunowanie...")
tuner.search(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop_tuning, reduce_lr_tuning, tensorboard_callback], verbose=2)

best_models = tuner.get_best_hyperparameters(num_trials=5)
best_hps = best_models[0]

print("\n🏆 Najlepszy model:")
print(f"units_1={best_hps.get('units_1')}, units_2={best_hps.get('units_2')}")
print(f"dropout_1={best_hps.get('dropout_1'):.2f}, dropout_2={best_hps.get('dropout_2'):.2f}")
print(f"learning_rate={best_hps.get('learning_rate'):.6f}, l2_reg={best_hps.get('l2_reg'):.6f}")
print(f"unfreeze_layers={best_hps.get('unfreeze_layers')}")

In [ ]:
model_image = tuner.hypermodel.build(best_hps)
model_image.summary()

## 18. Trening finalnego modelu

**Czas:** ~15-20 min (bez GPU) / ~4-7 min (z GPU)  
**Epoki:** 20 z early stopping

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint('saved_models/image_model_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

history = model_image.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[checkpoint, early_stop, reduce_lr])

print(f"✅ Najlepsza val_accuracy: {max(history.history['val_accuracy']):.2%}")

## 19. Wykresy treningu

In [ ]:
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
difference = final_train_acc - final_val_acc

print(f"Train: {final_train_acc:.2%}, Val: {final_val_acc:.2%}, Różnica: {difference:.2%}")

if difference < 0.05:
    print("🟢 GOOD FIT")
elif difference < 0.10:
    print("🟡 SLIGHT OVERFITTING")
else:
    print("🔴 OVERFITTING")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], marker='o', label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], marker='s', label='Validation', linewidth=2)
ax1.set_title('Accuracy podczas treningu', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoka', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'], marker='o', label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], marker='s', label='Validation', linewidth=2)
ax2.set_title('Loss podczas treningu', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoka', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 20. Ewaluacja na zbiorze testowym

In [ ]:
model_image = keras.models.load_model('saved_models/image_model_best.h5')
test_loss, test_acc = model_image.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.2%}")

## 21. Confusion Matrix

In [ ]:
y_pred = []
y_true = []

for images, labels in test_ds:
    preds = model_image.predict(images, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', xticklabels=emotions_list, yticklabels=emotions_list)
plt.title('Confusion Matrix - Model Obrazowy (znormalizowana)', fontsize=16, fontweight='bold')
plt.ylabel('Prawdziwa Emocja', fontsize=12)
plt.xlabel('Przewidziana Emocja', fontsize=12)
plt.tight_layout()
plt.show()

report = classification_report(y_true, y_pred, target_names=emotions_list, output_dict=True)
for emotion in emotions_list:
    m = report[emotion]
    print(f"{emotion}: Precision={m['precision']:.2%}, Recall={m['recall']:.2%}, F1={m['f1-score']:.2%}")
print(f"\nAccuracy: {report['accuracy']:.2%}")

## 22. Zapisanie finalnego modelu

In [ ]:
import json

model_image.save('saved_models/image_model_final.h5')
with open('saved_models/image_emotion_map.json', 'w') as f:
    json.dump(id_to_emotion, f)

print("✅ Model zapisany")